# Captura de imágenes
Libreta usada exclusivamente para capturar imagenes bloqueando el rostro de las personas en video

In [6]:
import cv2

entrada = "video_original.mp4"
salida = "video_anonimizado.mp4"

# Detector frontal incluido con OpenCV.
# No detectará todos los rostros en todas las condiciones.
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

cap = cv2.VideoCapture(1)
if not cap.isOpened():
    raise RuntimeError(f"No se pudo abrir el video: {entrada}")

fps = cap.get(cv2.CAP_PROP_FPS) or 25
ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
alto = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

writer = cv2.VideoWriter(
    salida,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (ancho, alto)
)
if not writer.isOpened():
    cap.release()
    raise RuntimeError("No se pudo crear el video de salida.")

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        gris = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        rostros = face_cascade.detectMultiScale(
            gris,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(30, 30)
        )

        # Margen alrededor del rostro para cubrir algo más que la cara.
        for (x, y, w, h) in rostros:
            margen_x = int(0.15 * w)
            margen_y = int(0.20 * h)
            x1 = max(0, x - margen_x)
            y1 = max(0, y - margen_y)
            x2 = min(ancho, x + w + margen_x)
            y2 = min(alto, y + h + margen_y)

            # Máscara opaca
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 0), thickness=-1)

        writer.write(frame)

        # Vista en tiempo real del video ya procesado
        cv2.imshow("Video anonimizado - presiona q para salir", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

finally:
    cap.release()
    writer.release()
    cv2.destroyAllWindows()

print("Procesamiento terminado.")


Procesamiento terminado.


In [18]:
import cv2
import time
from pathlib import Path
from datetime import datetime

# Configuración
CAMERA_INDEX = 1
COOLDOWN_SECONDS = 5.0
OUTPUT_DIR = Path("raw")

# Por precaución, no guardar una captura si el detector no encuentra
# ningún rostro. Esto no garantiza que haya detectado todos los rostros.
SAVE_ONLY_IF_FACE_DETECTED = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)
if face_cascade.empty():
    raise RuntimeError("No se pudo cargar el detector de rostros de OpenCV.")

cap = cv2.VideoCapture(CAMERA_INDEX)
if not cap.isOpened():
    raise RuntimeError("No se pudo abrir la webcam.")

last_attempt = time.monotonic() - COOLDOWN_SECONDS
status = "Esperando captura..."

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            print("No se pudo leer un fotograma de la webcam.")
            break

        height, width = frame.shape[:2]

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(
            gray,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(30, 30)
        )

        # Crear la imagen anonimizada; no se guarda el fotograma original.
        anonymized = frame.copy()

        for (x, y, w, h) in faces:
            # Ampliar la máscara alrededor del rostro detectado.
            margin_x = int(0.25 * w)
            margin_y = int(0.25 * h)

            x1 = max(0, x - margin_x)
            y1 = max(0, y - margin_y)
            x2 = min(width, x + w + margin_x)
            y2 = min(height, y + h + margin_y)

            # Rectángulo negro opaco.
            cv2.rectangle(
                anonymized,
                (x1, y1),
                (x2, y2),
                (0, 0, 0),
                thickness=-1
            )

        now = time.monotonic()

        # Intentar guardar una nueva captura al cumplirse el intervalo.
        if now - last_attempt >= COOLDOWN_SECONDS:
            last_attempt = now

            if len(faces) > 0 or not SAVE_ONLY_IF_FACE_DETECTED:
                filename = datetime.now().strftime(
                    "captura_%Y%m%d_%H%M%S_%f.jpg"
                )
                output_path = OUTPUT_DIR / filename

                saved = cv2.imwrite(
                    str(output_path),
                    anonymized,
                    [cv2.IMWRITE_JPEG_QUALITY, 95]
                )
                status = (
                    f"Guardada: {filename}"
                    if saved
                    else "Error al guardar la captura"
                )
            else:
                status = "No guardada: no se detectó ningún rostro"

        # Mostrar una copia con el estado; el texto no se añade al archivo.
        preview = anonymized.copy()
        cv2.putText(
            preview,
            status,
            (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (0, 255, 0),
            2
        )
        cv2.imshow("Vista anonimizada — pulsa q para salir", preview)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

finally:
    cap.release()
    cv2.destroyAllWindows()


## Usando script

In [15]:
from pathlib import Path
import sys

# Busca el script desde la raíz del repositorio o desde DataCapture.
_candidates = (Path.cwd() / "DataCapture", Path.cwd())
_module_dir = next(
    (path.resolve() for path in _candidates if (path / "capture_wo_pii.py").is_file()),
    None,
)
if _module_dir is None:
    raise FileNotFoundError("No se encontró DataCapture/capture_wo_pii.py desde el directorio actual.")
sys.path.insert(0, str(_module_dir))

from capture_wo_pii import capture_video

# Cambia source=1 para otra cámara. 's' guarda una captura enmascarada; 'q' termina.
capture_video(source=1, padding=0.30)

Showing anonymized video. Press 's' to save a masked image; 'q' or Esc to quit.
Saved anonymized image: captures_wo_pii\capture_20261003_160254_463891.jpg
Saved anonymized image: captures_wo_pii\capture_20261003_160255_951299.jpg
